In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr
from tigramite import plotting as tp


# Generate Data

![](DAG1.png "DAG")

In [2]:
def generateData(T, var_names, uniform = True, interval = (0,10), seed=42):
    np.random.seed(seed)
    # data = np.random.randn(T, len(var_names))
    data = np.random.uniform(interval[0], interval[1], (T, len(var_names))) if uniform else np.random.randn(T, len(var_names))
    for t in range(1, T):
        data[t, 0] += 0.6*data[t-1, 1]**2
        data[t, 2] += 0.3*data[t-2, 1]**2
    df = pp.DataFrame(data, var_names=var_names)
    return df

def generateDataMul(T, var_names, uniform = True, interval = (0,10), seed=42):
    np.random.seed(seed)
    # data = np.random.randn(T, len(var_names))
    data = np.random.uniform(interval[0], interval[1], (T, len(var_names))) if uniform else np.random.randn(T, len(var_names))
    for t in range(1, T):
        data[t, 0] *= 0.2*data[t-1, 1]
        data[t, 2] *= 0.3*data[t-2, 1]  
    df = pp.DataFrame(data, var_names=var_names)
    # df = pp.DataFrame(np.log(data), var_names=var_names)
    return df

# Additive with PCMCI

In [3]:
T = 3000
max_lag = 2
var_names = [r'$X^0$', r'$X^1$', r'$X^2$']
pc_df = generateData(T, var_names, uniform=True, interval=(0,10), seed=42)

pcmci_parcorr = PCMCI(dataframe = pc_df, cond_ind_test = ParCorr(plot = False, bias = True, degree = 2), verbosity = 0)
pcmci_results = pcmci_parcorr.run_pcmci(tau_min = 1, tau_max = max_lag, pc_alpha = .05, alpha_level = .001)

# tp.plot_timeseries(dataframe=pc_df); plt.show()

# tau_max = 4
# original_correlations = pcmci_parcorr.get_lagged_dependencies(tau_max=tau_max, val_only=True)['val_matrix']
# lag_func_matrix = tp.plot_lagfuncs(val_matrix=original_correlations, setup_args={'var_names':var_names, 
#                                     'x_base':1, 'y_base':1}); plt.show()

tp.plot_graph(val_matrix=pcmci_results['val_matrix'], graph=pcmci_results['graph'], var_names=var_names, show_colorbar=False);

# matrix = tp.setup_density_matrix(N=pc_df.N, var_names=pc_df.var_names)
# matrix.add_densityplot(dataframe=pc_df, matrix_lags=None)
matrixS = tp.setup_scatter_matrix(N=pc_df.N, var_names=pc_df.var_names)
matrixS.add_scatterplot(dataframe=pc_df, matrix_lags=None)

TypeError: CondIndTest.__init__() got an unexpected keyword argument 'plot'

# PC

In [ ]:
graph = np.zeros_like(pcmci_results['graph'])

for child, y in pcmci_parcorr.all_parents.items():
    for a in y:
        lag = abs(a[1])
        parent = a[0]
        graph[(parent,child,lag)] = '-->'

tp.plot_graph(val_matrix=np.ones_like(pcmci_results['val_matrix'])*.85, graph=graph, var_names=var_names, show_colorbar=False);


# Multiplicative with PCMCI

In [ ]:
T = 1000
max_lag = 5
var_names = [r'$X^0$', r'$X^1$', r'$X^2$']
df_mul = generateDataMul(T, var_names, uniform=True, interval=(0,1), seed=42)

pcmci_parcorr = PCMCI(dataframe = df_mul, cond_ind_test = ParCorr(plot = False, bias = False, degree = 1, interaction_only=False), verbosity = 0)
pcmci_results = pcmci_parcorr.run_pcmci(tau_min = 1, tau_max = max_lag, pc_alpha = .05, alpha_level = .001)

# tp.plot_timeseries(dataframe=df_mul); plt.show()

# tau_max = 4
# original_correlations = pcmci_parcorr.get_lagged_dependencies(tau_max=tau_max, val_only=True)['val_matrix']
# lag_func_matrix = tp.plot_lagfuncs(val_matrix=original_correlations, setup_args={'var_names':var_names, 
#                                     'x_base':1, 'y_base':1}); plt.show()

tp.plot_graph(val_matrix=pcmci_results['val_matrix']*100, graph=pcmci_results['graph'], var_names=var_names, show_colorbar=False);

# matrix = tp.setup_density_matrix(N=df_mul.N, var_names=df_mul.var_names)
# matrix.add_densityplot(dataframe=df_mul, matrix_lags=None)
# matrixS = tp.setup_scatter_matrix(N=df_mul.N, var_names=df_mul.var_names)
# matrixS.add_scatterplot(dataframe=df_mul, matrix_lags=None)